# Case 6 · Should we cut prices? (Part 2)
**BUS-G 492 · Causal decisions with AI**

In [ ]:
# --- Setup: run this cell first -----------------------------------------------
import os
DATA_URL = "https://raw.githubusercontent.com/boyoung-seo/492-coding/main/cases/case6_iv_pricing/data/"
for candidate in ["data/", "cases/case6_iv_pricing/data/", "../../../cases/case6_iv_pricing/data/", "../../../../cases/case6_iv_pricing/data/"]:
    if os.path.exists(candidate + "store_weeks.csv"):
        DATA = candidate
        break
else:
    DATA = DATA_URL
print("Reading data from:", DATA)

## Stage 5 · Something you didn't know

> **From:** Marcus Oyelaran, Regional Manager, Midwest
> **Re:** how we actually set prices
>
> Every Monday at 7am we get a store-level **demand forecast** from the analytics vendor. When it says a big week is coming, I nudge the price **up**; slow week, I nudge it **down**. All the regional managers do it; it's in the pricing playbook. The forecast isn't stored anywhere.

HQ also found two files:

- **`supplier_costs.csv`**: the wholesale cost we paid each week in each region. It changes when regional supplier contracts are renegotiated and with fuel surcharges. Managers mark up from this cost. Shoppers never see it.
- **`price_test_log.csv`**: in spring 2025, 8 stores ran a **12-week price test**: each week, prices were set **at random** to $9.99, $11.99 or $13.99, ignoring the forecast. Those store-weeks are already in your data.

### Load the data

This is a new notebook, so nothing from Part 1 is loaded yet. Run this cell to load the Part 1 data again, then the next cell for the new files.

In [ ]:
%pip install -q linearmodels   # not preinstalled in Colab
import statsmodels.formula.api as smf
import pandas as pd, numpy as np
df = pd.read_csv(DATA + "store_weeks.csv", parse_dates=["week"])
print(df.shape)
df.head()

In [ ]:
costs = pd.read_csv(DATA + "supplier_costs.csv", parse_dates=["week"])
test = pd.read_csv(DATA + "price_test_log.csv", parse_dates=["week"])
print(costs.shape, test.shape)

## Your tasks

Use AI again, but this time **tell it the whole story**: how the data were generated, and what the new information is. Each step ends with a table: fill it in (double-click the cell to type) before moving on.

### Step 1 · The randomized price test

In 8 stores for 12 weeks, prices were set **at random** to $9.99, $11.99 or $13.99. Merge the test log onto the data (on `store` and `week`) to keep only those store-weeks. Regress **log units** on **log price** with store effects: the slope is the **price elasticity** (the % change in units when the price rises 1%).

| Data used | Price elasticity |
|---|---|
| All store-weeks, no controls (as in Part 1) | |
| Random test weeks only (ETT for the test stores) | |

Why can you trust the second number? What is its weakness?

In [ ]:
import numpy as np
df["lq"], df["lp"] = np.log(df.units_sold), np.log(df.price)
t = df.merge(test, on=["store", "week"])
# Your code here (hint: smf.ols("lq ~ lp", df) and smf.ols("lq ~ lp + C(store)", t))


### Step 2 · Instrumental variables: wholesale cost

Wholesale cost moves our price (managers mark up from cost) but shoppers never see it. The code below runs **two-stage least squares**: stage 1 predicts log price from log cost; stage 2 relates log units to that predicted price. Fill in:

| | Value |
|---|---|
| First-stage F statistic (strength; should be well above 10) | |
| Price elasticity (IV) | |

Name one way wholesale cost could affect sales **other than** through our price.

In [ ]:
from linearmodels.iv import IV2SLS
d = df.merge(costs, on=["region", "week"])
d["lcost"] = np.log(d.wholesale_cost)
iv = IV2SLS.from_formula("lq ~ 1 + promotion + holiday_week + avg_temp_f + C(store) + [lp ~ lcost]", d).fit()
print("elasticity:", round(iv.params["lp"], 2))
print("first-stage F:", round(float(iv.first_stage.diagnostics["f.stat"].iloc[0])))


### Step 3 · Compare the estimates

| Method | Price elasticity |
|---|---|
| All data, no controls | |
| All data, + store and week fixed effects | |
| Random price test | |
| IV: wholesale cost | |

Why do the first two have the wrong sign?

In [ ]:
# Your code here (hint: smf.ols("lq ~ lp + promotion + avg_temp_f + C(store) + C(week)", df))


### Step 4 · From elasticity to price

With elasticity e (e < −1), the profit-maximizing price is **cost × e ÷ (e + 1)**, with cost = $6.00.

| Elasticity used | e | Best price |
|---|---|---|
| Random price test | | |
| IV | | |

Predicted Q4 units ≈ 40 stores × 13 weeks × (average weekly bags) × (new price ÷ $12)^e. Predicted profit = (price − $6) × units.

In [ ]:
# Your code here


### Decide again

In [ ]:
TEAM = ""                  # e.g. "Team 3: Ana & Ben"  (use the same name every time)
PRICE = None               # one chain-wide shelf price for Q4 ($)  (e.g. 12.49)
PREDICTED_PROFIT = None    # in dollars, as a number, e.g. 150000
JUSTIFICATION = """
Three sentences for the decision maker: what you found, what you recommend, and why.
"""

# Stops with a message if anything is still blank, so you don't submit an empty field.
blank = [name for name, value in [("TEAM", TEAM), ("PRICE", PRICE), ("PREDICTED_PROFIT", PREDICTED_PROFIT)] if value in (None, "")]
if blank:
    raise ValueError("Fill these in before submitting: " + ", ".join(blank))
PREDICTED_PROFIT = float(PREDICTED_PROFIT)   # must be a number
print("Enter exactly these values in the submission form:")
print()
print(f"team:              {TEAM}")
print("case:              6")
print("round:             2  (Part II)")
print(f"decision:          {PRICE}")
print(f"predicted_profit:  {PREDICTED_PROFIT:.0f}")

## Submit (round 2)

**[Open the submission form](https://docs.google.com/forms/d/e/1FAIpQLSf-87eDsEHRVat_a4le91sC7bfZFel9fAHNmMtHVuGJzqaHOQ/viewform)** and enter the values printed above, exactly as shown.

- **Predicted profit:** a number such as `150000` (`$150,000` and `150K` also work, but not words). The decision is a price in dollars and cents, e.g. `11.49`.
- **Case:** `6`.  **Round:** `2` (Part II).
- **Submissions are final.** You can't change an answer after you submit, so check the printed values first. If your team submits more than once for the same round, only the **first** submission counts.
- Use the **same team name** in every round.

## Reflection (submit)

1. Your Part 1 analysis may have fit the data very well. Why was it still the wrong tool for this decision?
2. In one sentence: what was the difference between **predicting** the outcome and **deciding** what to do?
3. Name one other business decision where the same trap could appear.